In [ ]:
###风险链条
import json
import re
import requests
import os
from typing import List
import networkx as nx
from pyvis.network import Network


# =====================================================
# 事件发展 + 风险演化 Prompt（只输出核心三元组）
# =====================================================

MAIN_SYSTEM_PROMPT = """
你是一个事件演化与风险传播分析专家。

任务：

1. 提取事件发展过程中的因果关系
2. 拆解事件演化链条
3. 识别风险扩散路径
4. 将复杂过程拆分为最小因果单元
5. 因果事件用简洁的动名词短语表达

输出必须为 JSON 数组，每个元素包含：

{
  "cause": "...",
  "relationship_type": "...",
  "effect": "..."
}

仅输出 JSON，不允许解释说明。
"""

MAIN_USER_PROMPT = """
请分析以下新闻文本：

1. 提取完整事件发展因果链
2. 构建清晰的因果推进关系
3. 拆分为最小因果单元

文本如下：
"""

STANDARDIZE_SYSTEM_PROMPT = """
你是知识图谱标准化专家。

任务：

1. 合并同义事件
2. 统一事件表达方式
3. 保持因果方向不变

输出仍为 JSON 数组格式
"""


# =====================================================
# DeepSeek API 调用
# =====================================================

def call_llm(
    model: str,
    api_key: str,
    user_prompt: str,
    system_prompt: str,
    base_url: str = "https://api.deepseek.com/v1",
    temperature: float = 0,
    max_tokens: int = 2000,
):
    url = f"{base_url}/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}", 
        "Content-Type": "application/json",
    }

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()

    return response.json()["choices"][0]["message"]["content"]


# =====================================================
# 文本分块
# =====================================================

def chunk_text(text: str, chunk_size: int = 1200) -> List[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end
    return chunks


# =====================================================
# 提取 JSON
# =====================================================

def extract_json(text: str):
    json_pattern = r"\[.*?\]"
    match = re.search(json_pattern, text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group())
    except:
        return None


# =====================================================
# 单 chunk 抽取
# =====================================================

def extract_from_chunk(text: str, api_key: str):
    user_prompt = MAIN_USER_PROMPT + f"```\n{text}\n```"

    raw_output = call_llm(
        model="deepseek-chat",
        api_key=api_key,
        user_prompt=user_prompt,
        system_prompt=MAIN_SYSTEM_PROMPT,
    )

    triples = extract_json(raw_output)
    if not triples:
        return []

    clean = []
    for t in triples:
        if (
            isinstance(t, dict)
            and "cause" in t
            and "relationship_type" in t
            and "effect" in t
        ):
            clean.append(t)
    return clean


# =====================================================
# 标准化
# =====================================================

def standardize_entities(triples, api_key):
    if not triples:
        return []

    user_prompt = f"""
以下因果三元组请进行标准化：

{json.dumps(triples, ensure_ascii=False, indent=2)}

输出格式保持不变。
"""

    raw_output = call_llm(
        model="deepseek-chat",
        api_key=api_key,
        user_prompt=user_prompt,
        system_prompt=STANDARDIZE_SYSTEM_PROMPT,
    )

    normalized = extract_json(raw_output)
    return normalized if normalized else triples


# =====================================================
# 构建图
# =====================================================

def build_graph(all_results):
    G = nx.DiGraph()
    for _, triples in all_results.items():
        for t in triples:
            G.add_edge(t["cause"], t["effect"], label=t["relationship_type"])
    return G


# =====================================================
# 自动识别最直接的主要后果链（基于贪心算法寻找最高频路径）
# =====================================================

def identify_main_risk_chain(G):
    if len(G.nodes()) == 0:
        return []
        
    # 1. 依然通过节点总度数（入度+出度）找到“震中”核心节点
    scores = {node: G.out_degree(node) + G.in_degree(node) for node in G.nodes()}
    if not scores:
        return []
    
    start_node = max(scores, key=scores.get)
    
    # 2. 贪心搜索：从核心点出发，只走“最强”的那条路
    main_chain = [start_node]
    current_node = start_node
    visited = {start_node} # 防止图中存在环导致死循环

    while True:
        # 获取当前节点的所有后续节点
        successors = list(G.successors(current_node))
        if not successors:
            break
            
        # 在所有后续节点中，选择“出度”最高的节点作为下一个“最直接”的影响后果
        # 出度高代表该后果会继续引发更多连锁反应，是传导链上的关键节点
        next_node = max(successors, key=lambda n: G.out_degree(n))
        
        if next_node in visited:
            break
            
        main_chain.append(next_node)
        visited.add(next_node)
        current_node = next_node

    return main_chain


# =====================================================
# 计算最长路径（风险传播最长链）
# =====================================================

def compute_longest_path(G):
    if not nx.is_directed_acyclic_graph(G):
        G = nx.DiGraph(nx.algorithms.dag.transitive_reduction(G))

    try:
        longest_path = nx.dag_longest_path(G)
    except:
        longest_path = []

    return longest_path


# =====================================================
# 可视化链条
# =====================================================

def visualize_path(G, path_nodes, filename):
    # 1. 防御性检查：如果没有节点，直接生成一个提示文件或不生成，避免报错
    if not path_nodes or len(path_nodes) == 0:
        print(f"⚠️ 警告: 路径为空，无法生成可视化文件 {filename}")
        # 可选：创建一个简单的 HTML 提示文件
        with open(filename, "w", encoding="utf-8") as f:
            f.write("<html><body><h1>无风险链条数据可展示</h1></body></html>")
        return

    if len(G.nodes()) == 0:
        print(f"⚠️ 警告: 图为空，无法生成可视化文件 {filename}")
        with open(filename, "w", encoding="utf-8") as f:
            f.write("<html><body><h1>图谱数据为空</h1></body></html>")
        return

    print(f"🎨 正在生成可视化: {filename} (节点数: {len(path_nodes)})")
    
    # 2. 初始化网络，明确设置 directed=True
    net = Network(height="800px", width="100%", directed=True, notebook=False)
    
    # 优化布局算法，barnes_hut 对于大图更稳定
    net.barnes_hut()

    # 将路径节点集合化以提高查找效率
    path_nodes_set = set(path_nodes)
    path_edges = set(zip(path_nodes, path_nodes[1:]))

    # 添加节点
    for node in G.nodes():
        # 如果节点在路径中，标红；否则标蓝
        color = "#ff4d4d" if node in path_nodes_set else "#97c2fc" 
        # 确保 label 是字符串，防止某些对象导致渲染问题
        net.add_node(node, label=str(node), color=color, title=str(node))

    # 添加边
    for source, target, data in G.edges(data=True):
        color = "#ff4d4d" if (source, target) in path_edges else "#cccccc"
        label = data.get("label", "")
        
        # 确保边连接的两个节点都已存在（虽然 add_edge 通常会自动添加，但显式确认更安全）
        net.add_edge(
            source,
            target,
            label=str(label),
            color=color,
            arrows="to",
            title=str(label)
        )

    # 3. 关键修复：强制 notebook=False 并捕获潜在异常
    try:
        # 在脚本模式下，show 方法等价于 write_html(name, open_browser=True)
        # 如果不想自动打开浏览器，可以直接用 net.write_html(filename, open_browser=False)
        net.show(filename)
        print(f"✅ 成功生成: {filename}")
    except Exception as e:
        print(f"❌ 生成 {filename} 时出错: {e}")
        # 降级方案：尝试直接写入
        try:
            net.write_html(filename, open_browser=False)
            print(f"✅ 通过 write_html 成功生成: {filename}")
        except:
            pass

# =====================================================
# 主程序
# =====================================================

if __name__ == "__main__":
    api_key = os.getenv("DEEPSEEK_API_KEY")
    if not api_key:
        raise ValueError("请设置 DEEPSEEK_API_KEY")

    texts_dir = "texts"
    all_results = {}

    for filename in os.listdir(texts_dir):
        if filename.endswith(".txt"):
            filepath = os.path.join(texts_dir, filename)
            with open(filepath, "r", encoding="utf-8") as f:
                text = f.read()

            chunks = chunk_text(text)
            triples_all = []

            for chunk in chunks:
                triples_all.extend(extract_from_chunk(chunk, api_key))

            triples_all = standardize_entities(triples_all, api_key)
            all_results[filename] = triples_all

    with open("all_triples.json", "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    print("✅ 因果三元组抽取完成")

    # 构建图
    G = build_graph(all_results)

    # 主风险链
    main_chain = identify_main_risk_chain(G)
    print(f"主风险链节点数量: {len(main_chain)}") # 确认这里不是 0
    visualize_path(G, main_chain, "main_risk_chain.html")
    print("✅ 主风险链已生成")

    # 最长风险路径
    longest_path = compute_longest_path(G)
    visualize_path(G, longest_path, "longest_risk_path.html")
    print("✅ 最长风险路径已生成")

✅ 因果三元组抽取完成
主风险链节点数量: 22
🎨 正在生成可视化: main_risk_chain.html (节点数: 22)
main_risk_chain.html
❌ 生成 main_risk_chain.html 时出错: 'NoneType' object has no attribute 'render'
✅ 通过 write_html 成功生成: main_risk_chain.html
✅ 主风险链已生成
🎨 正在生成可视化: longest_risk_path.html (节点数: 6)
longest_risk_path.html
❌ 生成 longest_risk_path.html 时出错: 'NoneType' object has no attribute 'render'
✅ 通过 write_html 成功生成: longest_risk_path.html
✅ 最长风险路径已生成
